# Language, Writing, and AI: Coastal Carolina Workshop

This notebook accompanies a 90-minute session for English students. You will complete two hands-on activities. The instructor drives a separate prediction demo on their own machine. No coding experience is required.

| Tier | Task | Format |
|---|---|---|
| 1 | Extract entities and likely actions | Hands-on |
| 2 | Evaluate text with rules and revise it | Hands-on |

Run cells in order with **Shift + Enter** or the play button. Keep a copy in Google Drive to save edits. Use only text you can share; all supplied examples are synthetic.

Copyright (c) 2026 David Young. Code cells: [MIT](https://github.com/dyoung8989/CoastCarolinaWorkshop/blob/main/LICENSE-MIT). Markdown explanations and standalone sample teaching text: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). See [licensing details](https://github.com/dyoung8989/CoastCarolinaWorkshop/blob/main/LICENSE.md) for scope and attribution. Third-party dependencies retain their own licenses.

## Step 0: Setup

Run these two cells first. Setup downloads the libraries and a small English model; allow a few minutes. If Colab requests a restart, restart the session and run from the top. If the download fails, pair with someone or follow the instructor's saved demo.

In [ ]:
# Install the versions used by this workshop into the Colab session.
%pip install spacy==3.8.16 weasel==1.0.0 typer==0.27.3 click==8.4.2 pandas==2.2.3 numpy==2.2.6
%pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

In [ ]:
import re
import spacy
import pandas as pd

nlp = spacy.load("en_core_web_sm")

print("Setup complete! You're ready for Tier 1.")

---
## Tier 1: Extraction

NLP tools can pull information from text, including names, key terms, and likely action items. We will review what the tools find and check for mistakes.

Here's a short sample procedure, the kind of thing a technical writer might produce:

In [ ]:
sample_doc_1 = """
Before you begin, make sure you have administrator access to the instance. First, navigate
to System Settings and select User Administration. You must verify that the user's role
includes "admin" before proceeding. Next, click Add User and enter the required fields:
username, email, and department. The system should automatically assign a default password,
but you can override this if needed. Please confirm the new account by clicking Save. If the
account does not appear in the list within 60 seconds, contact the IT Help Desk at extension
4521. Do not share admin credentials with other users under any circumstances.
"""

print(sample_doc_1)

### Try it: find the named things

spaCy can label names, organizations, quantities, and other entities it recognizes. Some labels will be wrong. Review the output against the original text.

In [ ]:
doc = nlp(sample_doc_1)

print("Named things spaCy found:\n")
for ent in doc.ents:
    print(f"  {ent.text!r:30}  ->  {ent.label_}")

### Try it: find the action items

We will look for sentences containing instruction words such as "must", "click", "please", and "do not". This is a keyword rule. It can miss instructions and flag sentences that are not instructions.

In [ ]:
action_keywords = ["must", "click", "please", "do not", "ensure", "verify", "confirm", "contact", "navigate", "select"]

def extract_actions(text):
    # Match whole words/phrases, so "mustard" does not match "must".
    pattern = r"\b(?:" + "|".join(re.escape(word) for word in action_keywords) + r")\b"
    return [sent.text.strip() for sent in nlp(text).sents
            if re.search(pattern, sent.text, flags=re.IGNORECASE)]

print("Likely action items (keyword guesses):\n")
for action in extract_actions(sample_doc_1):
    print(f"  • {action}")

**Your turn:** replace the paragraph below, keeping the triple quotes, then run the cell. It analyzes your new text directly. Challenge one entity label and look for an action the keyword list missed. The entity recognizer uses a trained model; the action detector uses rules.

In [ ]:
your_text = """
Please send the draft to the campus writing center. Maya will review it on Friday.
"""

print("Entities:")
for entity in nlp(your_text).ents:
    print(f"  {entity.text!r} -> {entity.label_}")
print("\nLikely actions:")
for action in extract_actions(your_text):
    print(f"  • {action}")

---
## Tier 2: Automated evaluation

Now we will review a few features of the writing and combine them into a document readiness score. The output includes flags with suggestions to consider.

The rules and thresholds below are teaching choices. The score does not check accuracy, completeness, or whether the text fits its audience.

The signals we'll combine:
- **Readability**: how hard the sentences are to read (via the Flesch Reading Ease score)
- **Passive voice**: what share of sentences use passive voice ("the form is submitted" vs. "you submit the form")
- **Sentence length**: average words per sentence
- **Structure**: does the document have headers or numbered steps at all?

In [ ]:
def count_syllables(word):
    """Rough syllable count: counts groups of vowels. Good enough for a readability estimate."""
    word = word.lower()
    groups = re.findall(r"[aeiouy]+", word)
    count = len(groups)
    if word.endswith("e") and count > 1:
        count -= 1
    return max(count, 1)


def flesch_reading_ease(text):
    """A simplified version of the classic Flesch Reading Ease formula. Higher = easier to read."""
    words = re.findall(r"[a-zA-Z']+", text)
    sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    num_words = max(len(words), 1)
    num_sentences = max(len(sentences), 1)
    num_syllables = sum(count_syllables(w) for w in words)
    return 206.835 - 1.015 * (num_words / num_sentences) - 84.6 * (num_syllables / num_words)


def score_doc(text):
    """Scores a document's readiness for a reader, 0-100, with specific flags."""
    if not isinstance(text, str) or not re.search(r"[a-zA-Z]", text):
        raise ValueError("Enter a paragraph with English words before scoring.")
    doc = nlp(text)
    sentences = list(doc.sents)
    num_sentences = max(len(sentences), 1)

    # Passive voice: spaCy tags the subject of a passive sentence as 'nsubjpass'
    passive_count = sum(1 for sent in sentences if any(tok.dep_ == "nsubjpass" for tok in sent))
    passive_pct = passive_count / num_sentences * 100

    # Average sentence length, in words
    avg_sentence_length = sum(len([t for t in sent if t.is_alpha]) for sent in sentences) / num_sentences

    # Readability: higher = easier (roughly 0-100 scale)
    readability = flesch_reading_ease(text)

    # Structure: any markdown-style header or numbered step?
    has_structure = bool(re.search(r"(^#{1,3}\s)|(^\s*\d+[.)]\s)", text, re.MULTILINE))

    flags = []
    if readability < 50:
        flags.append("Readability is low: try shorter sentences and simpler words.")
    if passive_pct > 30:
        flags.append(f"{passive_pct:.0f}% of sentences are passive: consider rewriting in active voice.")
    if avg_sentence_length > 25:
        flags.append("Average sentence length is high: break up long sentences.")
    if not has_structure:
        flags.append("No headers or numbered steps detected: consider adding structure.")

    score = 100
    score -= max(0, 60 - readability)
    score -= passive_pct * 0.5
    score -= max(0, avg_sentence_length - 20) * 2
    score -= 0 if has_structure else 15
    score = max(0, min(100, round(score)))

    return {
        "readiness_score": score,
        "readability": round(readability, 1),
        "passive_voice_pct": round(passive_pct, 1),
        "avg_sentence_length": round(avg_sentence_length, 1),
        "has_structure": has_structure,
        "flags": flags,
    }

print("score_doc() is ready.")

### Try it: score a messy document

In [ ]:
messy_doc = """
The configuration settings are modified by the administrator once the new environment has
been provisioned by the infrastructure team. It should be noted that the deployment process,
which involves several interdependent steps that must be completed in a specific order to
avoid conflicts with existing services, is not always fully understood by new team members,
and as a result, errors are frequently introduced during the initial setup phase, which can
then cascade into larger system failures if they are not caught early by the monitoring tools
that have been configured to alert the on-call engineer.
"""

result = score_doc(messy_doc)

print(f"Readiness score: {result['readiness_score']} / 100\n")
print(f"  Readability:          {result['readability']}")
print(f"  Passive voice:        {result['passive_voice_pct']}%")
print(f"  Avg sentence length:  {result['avg_sentence_length']} words")
print(f"  Has structure:        {result['has_structure']}\n")
print("Flags:")
for flag in result["flags"]:
    print(f"  \u26a0 {flag}")

### Compare a rewrite

This rewrite is shorter and more structured, but it also omits detail. Compare both the score and what a reader would learn. Does a higher score guarantee a better procedure? These rules cannot judge completeness or correctness.

In [ ]:
improved_doc = """
## Deployment Steps

1. The infrastructure team provisions the new environment.
2. The administrator updates the configuration settings.
3. Complete each step in order to avoid conflicts with existing services.
4. Monitoring tools alert the on-call engineer if an error occurs.
"""

result2 = score_doc(improved_doc)

print(f"Readiness score: {result2['readiness_score']} / 100  (was {result['readiness_score']} / 100)\n")
print(f"  Readability:          {result2['readability']}")
print(f"  Passive voice:        {result2['passive_voice_pct']}%")
print(f"  Avg sentence length:  {result2['avg_sentence_length']} words")
print(f"  Has structure:        {result2['has_structure']}")

### Your turn: compare your original and revision

Replace the example text inside the triple quotes for `original`, then write your edited version inside the triple quotes for `revision`. Keep both sets of triple quotes. Run the cell to score both texts and see their flags side by side in a table.

Change one issue at a time and rerun. Preserve the meaning: does the score reward an edit that also helps the intended reader? Passive voice is sometimes appropriate, and technical vocabulary may be necessary. A higher score does not guarantee a better document.


In [ ]:
# Replace each text block with your own non-sensitive writing.
original = """
Workshop registration must be completed by students before Friday. It should be
noted that confirmation of registration will be sent by the writing center after
the form has been reviewed, and students who have not received confirmation
within two days should contact the writing center.
"""

revision = """
## Register for the Workshop

1. Complete your workshop registration before Friday.
2. The writing center will review your form and send confirmation.
3. Contact the writing center if you do not receive confirmation within two days.
"""

original_result = score_doc(original)
revision_result = score_doc(revision)


def comparison_values(scores):
    return {
        "Readiness score (0–100)": scores["readiness_score"],
        "Readability (higher = easier)": scores["readability"],
        "Passive voice (%)": scores["passive_voice_pct"],
        "Average sentence length (words)": scores["avg_sentence_length"],
        "Has structure": scores["has_structure"],
        "Flags": "; ".join(scores["flags"]) if scores["flags"] else "No flags",
    }


comparison = pd.DataFrame({
    "Original": comparison_values(original_result),
    "Revision": comparison_values(revision_result),
})

# Show the complete flags rather than truncating long text in the table.
with pd.option_context("display.max_colwidth", None):
    display(comparison)


---
## Wrap-up

You used extraction and rule-based evaluation to examine writing and compare revisions. The instructor drives the separate prediction demo on their own machine.

Discuss: what did the tools miss? What knowledge did you bring as a reader and writer? What else would you check before deciding a revision is more useful?

Next step: apply one exercise to a public paragraph and record both its useful findings and its mistakes. See the repository's student guide and resources for follow-up links.
